In [1]:
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder().\
    config("spark.sql.autoBroadcastJoinThreshold", "-1")

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/08 09:55:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/08 09:55:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/08 09:55:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/08 09:55:25 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/11/08 09:55:25 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/11/08 09:55:25 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/11/08 09:55:25 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/11/

In [3]:
(sedona
    .read
    .format("binaryFile")
    .load(f"s3a://{bucket_name}/source_data/fdi_data")
    .selectExpr("RS_FromGeoTiff(content) AS rast")
    .createOrReplaceTempView("ffdi"))

(
    sedona
        .read
        .format("binaryFile")
        .load(f"s3a://{bucket_name}/source_data/world_population_raster")
        .selectExpr("RS_FromGeoTiff(content) AS rast")
        .createOrReplaceTempView("population")
)

Py4JJavaError: An error occurred while calling o56.load.
: java.nio.file.AccessDeniedException: s3a://ptokaj-sedona-book/source_data/fdi_data: org.apache.hadoop.fs.s3a.auth.NoAwsCredentialsException: SimpleAWSCredentialsProvider: No AWS credentials in the Hadoop configuration
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:212)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3799)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$exists$34(S3AFileSystem.java:4703)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:499)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:444)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AFileSystem.java:23

In [ ]:
sedona.sql(
    """
    SELECT 
        raster.tile as rast,
        raster.x,
        raster.y 
    FROM ffdi
    LATERAL VIEW RS_TileExplode(rast, 100, 100) raster
    """
).createOrReplaceTempView("fdi_tiles")

In [10]:
geometry = sedona.sql(
    """
    WITH pixelized AS (
        SELECT 
            RS_PixelAsPolygons(rast, 1) AS pixels,
            x,
            y
        FROM fdi_tiles
    ),
    classified AS (
        SELECT
            pixel.geom,
            x,
            y,
            CASE
                WHEN pixel.value > 50 THEN 'extreme'
                WHEN pixel.value > 25 THEN 'very high'
                WHEN pixel.value > 12 THEN 'high'
                WHEN pixel.value > 5 THEN 'moderate'
                WHEN pixel.value > 0 THEN 'low'
            END AS fire_danger_class
        FROM pixelized
        LATERAL VIEW explode(pixels) AS pixel
        WHERE pixel.value > 0 AND pixel.value < 255
    )
     SELECT
            ST_Union_Aggr(geom) AS geom,
            fire_danger_class
        FROM classified
        GROUP BY fire_danger_class, x, y
    """
).createOrReplaceTempView("fire_danger")

In [11]:
sedona.sql("SELECT * FROM fire_danger").show(5)

[Stage 6:>                                                          (0 + 1) / 1]

+--------------------+-----------------+
|                geom|fire_danger_class|
+--------------------+-----------------+
|MULTIPOLYGON (((-...|             high|
|MULTIPOLYGON (((7...|              low|
|MULTIPOLYGON (((3...|         moderate|
|MULTIPOLYGON (((7...|         moderate|
|MULTIPOLYGON (((-...|             high|
+--------------------+-----------------+
only showing top 5 rows



In [12]:
## SELECT  RS_ZonalStats(rast, 1, ST_CollectionExtrac(geom), 1, 'sum', true, false) AS population_sum
sedona.sql(
    
    """
    WITH intersection AS (
        SELECT 
            rast,
            ST_Buffer(ST_Intersection(RS_Envelope(rast), geom), -0.0001) AS geom,
            fire_danger_class
        FROM population AS p
        JOIN fire_danger AS f ON RS_Intersects(p.rast, f.geom)
    ),
    zonal_stats AS (
        SELECT 
            RS_ZonalStats(rast, geom, 1, 'sum') AS population_sum,
            fire_danger_class
        FROM intersection
    )
    SELECT 
        fire_danger_class,
        CAST(sum(population_sum) AS DECIMAL(38, 0)) AS population_sum
    FROM zonal_stats
    GROUP BY fire_danger_class

    """
).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[fire_danger_class#131], functions=[sum(population_sum#184)])
   +- Exchange hashpartitioning(fire_danger_class#131, 200), ENSURE_REQUIREMENTS, [plan_id=240]
      +- HashAggregate(keys=[fire_danger_class#131], functions=[partial_sum(population_sum#184)])
         +- Project [ **org.apache.spark.sql.sedona_sql.expressions.raster.RS_ZonalStats**   AS population_sum#184, fire_danger_class#131]
            +- RangeJoin rast#119: raster, geom#129: geometry, INTERSECTS,  **org.apache.spark.sql.sedona_sql.expressions.raster.RS_Intersects**
               :- Project [ **org.apache.spark.sql.sedona_sql.expressions.raster.RS_FromGeoTiff**   AS rast#119]
               :  +- Filter isnotnull( **org.apache.spark.sql.sedona_sql.expressions.raster.RS_FromGeoTiff**  )
               :     +- FileScan binaryFile [content#114] Batched: false, DataFilters: [isnotnull( **org.apache.spark.sql.sedona_sql.expressions.raster.RS_Fr